# 📸 GFPGAN Premium Enhancer

**Mejora tus fotos con inteligencia artificial directamente desde la nube.**

Este notebook te permite restaurar y mejorar fotos de baja resolución usando GFPGAN, aprovechando la GPU gratuita de Google Colab.

> ⚠️ **Importante:** Antes de empezar, activa la GPU gratuita:
> 1. Ve a **Entorno de ejecución** (Runtime) en el menú superior
> 2. Haz clic en **Cambiar tipo de entorno de ejecución** (Change runtime type)
> 3. Selecciona **T4 GPU** en "Acelerador de hardware"
> 4. Haz clic en **Guardar**

## Paso 1: Instalar dependencias 📦
Ejecuta esta celda para clonar el repositorio e instalar todo lo necesario. Toma ~2 minutos.

In [ ]:
# Clonar TU repositorio fork
!git clone https://github.com/Bitwolf89/GFPGAN.git
%cd GFPGAN

# Instalar dependencias
!pip install basicsr facexlib realesrgan gfpgan
!pip install -r requirements.txt
!python setup.py develop

print('✅ Instalación completada!')

## Paso 2: Descargar modelos pre-entrenados 🧠
Descargamos el modelo V1.4 (el más reciente y nítido).

In [ ]:
# Descargar modelo GFPGAN v1.4
!wget https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth -P experiments/pretrained_models

# (Opcional) Descargar otros modelos
# !wget https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth -P experiments/pretrained_models
# !wget https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/RestoreFormer.pth -P experiments/pretrained_models

print('✅ Modelo descargado!')

## Paso 3: Subir tus fotos 🖼️
Sube las fotos que quieras mejorar. Se guardarán en la carpeta `inputs/upload`.

In [ ]:
import os
from google.colab import files

# Crear carpeta para las fotos subidas
upload_folder = 'inputs/upload'
os.makedirs(upload_folder, exist_ok=True)

# Subir fotos (se abre un selector de archivos)
print('📂 Selecciona las fotos que quieras mejorar...')
uploaded = files.upload()

# Mover a la carpeta de inputs
for filename in uploaded.keys():
    os.rename(filename, os.path.join(upload_folder, filename))
    print(f'  ✅ {filename} subida correctamente')

print(f'\n🎉 {len(uploaded)} foto(s) lista(s) para mejorar!')

## Paso 4: ¡Mejorar las fotos! 🚀
Ejecuta la restauración. Puedes ajustar:
- **`-v 1.4`**: Versión del modelo (1.2, 1.3, 1.4 o RestoreFormer)
- **`-s 2`**: Escala de aumento (2 = doble resolución, hasta 4)
- **`--bg_upsampler realesrgan`**: Mejorar también el fondo (no solo la cara)

In [ ]:
# === CONFIGURACIÓN ===
VERSION = '1.4'        # Opciones: '1', '1.2', '1.3', '1.4', 'RestoreFormer'
ESCALA = 2             # 2 = doble resolución, 4 = cuádruple
MEJORAR_FONDO = True   # True para mejorar también el fondo con Real-ESRGAN
# =====================

bg_flag = '--bg_upsampler realesrgan' if MEJORAR_FONDO else '--bg_upsampler none'

!python inference_gfpgan.py \
    -i inputs/upload \
    -o results \
    -v {VERSION} \
    -s {ESCALA} \
    {bg_flag}

print('✅ ¡Fotos mejoradas!')

## Paso 5: Ver resultados 👀
Compara las fotos originales con las mejoradas lado a lado.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import glob
import os

# Buscar resultados
restored_imgs = sorted(glob.glob('results/restored_imgs/*'))
input_imgs = sorted(glob.glob('inputs/upload/*'))

for inp_path, res_path in zip(input_imgs, restored_imgs):
    # Leer imágenes
    img_input = cv2.imread(inp_path)
    img_input = cv2.cvtColor(img_input, cv2.COLOR_BGR2RGB)
    img_result = cv2.imread(res_path)
    img_result = cv2.cvtColor(img_result, cv2.COLOR_BGR2RGB)

    # Mostrar comparación
    fig, axes = plt.subplots(1, 2, figsize=(18, 9))
    axes[0].imshow(img_input)
    axes[0].set_title('🖼️ Original', fontsize=16)
    axes[0].axis('off')
    axes[1].imshow(img_result)
    axes[1].set_title('✨ Mejorada', fontsize=16)
    axes[1].axis('off')
    fig.suptitle(os.path.basename(inp_path), fontsize=14, y=0.02)
    plt.tight_layout()
    plt.show()

print(f'🌟 Se procesaron {len(restored_imgs)} foto(s)')

## Paso 6: Descargar resultados 📥
Descarga todas las fotos mejoradas como un archivo ZIP.

In [ ]:
import shutil
from google.colab import files

# Crear ZIP con todos los resultados
shutil.make_archive('fotos_mejoradas', 'zip', 'results')

# Descargar el ZIP
files.download('fotos_mejoradas.zip')

print('🎉 ¡Descarga iniciada! Revisa tu carpeta de descargas.')

---
## 📚 Tips

| Modelo | Fortaleza | Mejor para |
|--------|-----------|------------|
| **v1.4** | Más detalles, mejor identidad | Uso general ⭐ |
| **v1.3** | Resultados más naturales | Fotos muy dañadas |
| **v1.2** | Más nítido, con maquillaje | Retratos |
| **RestoreFormer** | Alternativa con otra arquitectura | Experimentar |

> 💡 Si la foto sale muy artificial, prueba con `-w 0.7` (agrega el parámetro weight en el Paso 4)

---
*Fork de [TencentARC/GFPGAN](https://github.com/TencentARC/GFPGAN) • Mejorado por Bitwolf89*